# Prerocessing Data Sample

#### Library

In [ ]:
import pandas as pd
import string
import re
string.punctuation
import numpy as np
import Sastrawi
import gensim

import nltk
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist


### Dataset

In [ ]:
df_bindo = pd.read_excel('D:\kuliah\THE ONLY TA THINGS\DATA\soal-bahasaindonesia-sample.xlsx')
df_bindo.head()

In [ ]:
df_bindo.info()

In [ ]:
df_bindo.dropna(axis=1)
df_bindo.drop_duplicates(inplace=True)
df_bindo.head()

In [ ]:
cols = ['NO', 'id', 'Unnamed: 6']
df = df_bindo.drop(cols, axis=1)

In [ ]:
df.dropna()
df.head()


## CLEANSING DATA 

In [ ]:

# # REMOVE PUNCTUATION
# def remove_punctuation(text):
#     return text.translate(str.maketrans("","",string.punctuation))

# df['cleansing'] = df['content'].apply(remove_punctuation)

# # REMOVE NUMBER
# def remove_number(text):
#     return re.sub(r"\d+", "", text)

# df['cleansing'] = df['cleansing'].apply(remove_number)

# # REMOVE SPESIFIC CONTENT
# def remove_sp_content(text):
#     # remove link, hashtag
#     text = ' '.join(re.sub("([@#][A-Za-z0-9]+)|(\w+:\/\/\S+)"," ", text).split())
#      # remove non ASCII (emoticon, chinese word, .etc)
#     text = text.encode('ascii', 'replace').decode('ascii')
#      # remove tab, new line, ans back slice
#     text = text.replace('\\t'," ").replace('\\n'," ").replace('\\u'," ").replace('\\',"")
#     # remove mention @
#     text = re.sub(r"[@][\w_-]+","", text)
#     # remove incomplete URL
#     return text.replace("http://", " ").replace("https://", " ")

# df['cleansing'] = df['cleansing'].apply(remove_sp_content)

# #remove whitespace leading & trailing
# def remove_whitespace_LT(text):
#     return text.strip()

# df['cleansing'] = df['cleansing'].apply(remove_whitespace_LT)

# #remove multiple whitespace into single whitespace
# def remove_whitespace_multiple(text):
#     return re.sub('\s+',' ',text)

# df['cleansing'] = df['cleansing'].apply(remove_whitespace_multiple)

# # remove single char
# def remove_singl_char(text):
#     return re.sub(r"\b[a-zA-Z]\b", "", text)

# df['cleansing'] = df['cleansing'].apply(remove_singl_char)

# df.loc[0:300,['content', 'cleansing']]

In [ ]:
def cleaning(data):
    data = re.sub(r"\n", " ",data) # menghilangkan tanda baca
    data = re.sub(r"\r", " ",data)
    data = re.sub(r"&amp;", " ",data)
    data = re.sub(r"&gt;", " ",data)
    data = re.sub(r"&lt;", " ",data)
    data = re.sub(r"&quot;", " ",data)
    data = re.sub(r"\u00a0", " ",data) 
    data = re.sub(r'''(?i)\b((?:https|http?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'".,<>???????]))''', "",data)
    data = re.sub(r"'s","",data)
    data = re.sub(r"can't","can not",data) # mengubah kata
    data = re.sub(r"ain't","am not",data)
    data = re.sub(r"won't","will not",data)
    data = re.sub(r"n't"," not",data)
    data = re.sub(r"'re"," are",data)    
    data = re.sub(r"[-_<>*#$@()\"/%&;:<>{}`+=~|.!?,']", " ", data) #hapus simbol
    data = re.sub(r"[^a-zA-Z]", " ",data) # menghilangkan angka
    data = re.sub("\s\s+", " ",data) # menghilangkan double spasi
    return data

In [ ]:
# menambahkan kolom baru yang berisikan reviewText yang sudah bersih dari unwanted character
df['cleansing'] = df['question'].apply(lambda x: cleaning(x))
df.head()

In [ ]:
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
display(df.loc[0:100,['question', 'cleansing']])

## CASE FOLDING

In [ ]:
df['case_folding'] = df['cleansing'].str.lower()
df.loc[0:300,['cleansing', 'case_folding']]

In [ ]:
# menambahkan kolom baru yang berisikan reviewText yang sudah bersih dari unwanted character
df['no_unwanted'] = df['case_folding'].apply(lambda x: cleaning(x))
df.head()

##  TOKENIZATION

In [ ]:
# # NLTK word rokenize 

nltk.download('punkt')
def tokenize(text):
    return word_tokenize(text);

df['tokenize'] = df['case_folding'].apply(tokenize);
df.loc[0:300,['case_folding', 'tokenize']]

## Normalization

In [ ]:
slang_dictionary = pd.read_csv('D:\kuliah\THE ONLY TA THINGS\DATA\Ref\slang_dictionary.csv')
normalization = {}
for idx, row in slang_dictionary.iterrows():
    if row[0] not in normalization:
        normalization[row[0]] = row[1]

def normalized_method(document):
    return [normalization[term] if term in normalization else term for term in document]

df['normalization'] = df['tokenize'].apply(normalized_method)

df.loc[0:300,["tokenize", "normalization"]]


## Stopword removal

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords

# ----------------------- get stopword from NLTK stopword -------------------------------
# get stopword indonesia
list_stopwords = stopwords.words('indonesian')

# ---------------------------- manualy add stopword  ------------------------------------
# append additional stopword
list_stopwords.extend(["yg", "dg", "rt", "dgn", "ny", "d", 'klo', 
                       'kalo', 'amp', 'biar', 'bikin', 'bilang', 
                       'gak', 'ga', 'krn', 'nya', 'nih', 'sih', 
                       'si', 'tau', 'tdk', 'tuh', 'utk', 'ya', 
                       'jd', 'jgn', 'sdh', 'aja', 'n', 't', 
                       'nyg', 'hehe', 'pen', 'u', 'nan', 'loh', 'rt',
                       '&amp', 'yah', 'gtgt','ltlt'])

# ----------------------- add stopword from txt file ------------------------------------
# read txt stopword using pandas
txt_stopword = pd.read_csv("D:\kuliah\THE ONLY TA THINGS\DATA\Ref\stopwords.txt", names= ["stopwords"], header = None)

# convert stopword string to list & append additional stopword
list_stopwords.extend(txt_stopword["stopwords"][0].split(' '))

# ---------------------------------------------------------------------------------------

# convert list to dictionary
list_stopwords = set(list_stopwords)


#remove stopword pada list token
def stopwords_removal(words):
    #kata = [word for word in words if word not in list_stopwords]
    return [word for word in words if word not in list_stopwords]

df['stopwords'] = df['normalization'].apply(stopwords_removal) 

df.loc[0:300,['normalization', 'stopwords']]

## Stemming

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import swifter


# create stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# stemmed
def stemmed_wrapper(term):
    return stemmer.stem(term)

term_dict = {}

for document in df['stopwords']:
    for term in document:
        if term not in term_dict:
            term_dict[term] = ' '
            
for term in term_dict:
    term_dict[term] = stemmed_wrapper(term)

# apply stemmed term to dataframe
def get_stemmed_term(document):
    return [term_dict[term] for term in document]

df['stemming'] = df['stopwords'].swifter.apply(get_stemmed_term)

df.loc[0:300,['stopwords', 'stemming']]

In [ ]:
df['target'] = df['Label.1']
df.head()

## preprocessing result

In [ ]:
df_result_prepro = df[['question', 'stemming', 'Label.1']].copy(deep=True)
df_result_prepro

In [ ]:
df_result_prepro.isnull().sum()

## export data

In [ ]:
# df_result_prepro.to_csv(r'D:\kuliah\THE ONLY TA THINGS\DATA\cleaned_data_non-numeric_review.csv', index=False, header=True)

## word2vec

In [ ]:
df = pd.read_csv('D:\kuliah\THE ONLY TA THINGS\DATA\cleaned_data_review.csv')
df.head()

In [ ]:
df_result_prepro["clean_question"] = df_result_prepro["stemming"].apply(lambda x: " ".join(x))
df_result_prepro

In [ ]:
# from gensim.models import KeyedVectors

path = "../MODEL/idwiki_word2vec_100/idwiki_word2vec_100.model"
model = gensim.models.Word2Vec.load(path)
# model = KeyedVectors.load_word2vec_format(path, binary=True) 

In [ ]:
# Embbed each word with token from pre-trained model
X = []

for arr_stemming in df_result_prepro['stemming']:
    for token in arr_stemming:
        vectors = []
        if token in model.wv:
                vectors.append(model.wv[token])
        if vectors:
            # X.append(np.mean(vectors, axis=0))
            X = np.mean(vectors, axis=0)
        else:
            # X.append(np.zeros(model.vector_size))
            X = np.zeros(model.vector_size)
X = np.array(X)

In [ ]:
# split label to y
y = df_result_prepro['Label.1']

In [ ]:
# Split the data into training and testing sets
train_size = int(len(df_result_prepro) * 0.8)
X_train = X[:train_size]
y_train = y[:train_size]
X_test = X[train_size:]
y_test = y[train_size:]

## SVM Classifier

In [ ]:
from gensim.models import KeyedVectors
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from nltk.tokenize import word_tokenize

# X_train = np.array(X_train)
# # X_train = X_train.reshape(-1, 1)
# y_train = np.array(y_train)
# # y_train = X_train.reshape(-1, 1)

# Fine-tune the SVM classifier using a grid search over hyperparameters
best_score = 0
best_params = {}
for C in [0.1, 1, 10, 100]:
    for gamma in [0.1, 1, 10, 100]:
        for kernel in ['linear', 'rbf']:
            svm = SVC(C=C, gamma=gamma, kernel=kernel)
            svm.fit(X_train, y_train)
            y_pred = svm.predict(X_test)
            score = accuracy_score(y_test, y_pred)
            if score > best_score:
                best_score = score
                best_params = {'C': C, 'gamma': gamma, 'kernel': kernel}

# Train the SVM classifier on the entire dataset using the best hyperparameters
svm = SVC(**best_params)
svm.fit(X, y)

## SVM Classifier - Kiko

In [ ]:
# #K-Fold Cross Validation will iterate k times
# kFoldCrossValidation = KFold(n_splits=7, random_state=0, shuffle = True)

# actual_classes = np.empty([0], dtype=int)
# predicted_classes = np.empty([0], dtype=int)
    
# for train, test in kFoldCrossValidation.split(TFIDF_question, label):
#     #Initiate Train and Test Data then transform to TFIDF value. Then copy to new Train and Test variables. 
#     trainData, testData = TFIDF_question[train], TFIDF_question[test]
#     label = np.array(label)
#     trainData2, testData2 = label[train], label[test]
    
#     SVM = SVC(kernel = 'linear', C = 1)
#     model = SVM.fit(trainData, trainData2)
#     prediksi = model.predict(testData)
    
#     actual_classes = np.append(actual_classes, label[test])
#     predicted_classes = np.append(predicted_classes, model.predict(testData))
   
#     accuracy.append(accuracy_score(testData2, prediksi))

# conf_matrix = metrics.confusion_matrix(actual_classes, predicted_classes)
# svm_accuracy = metrics.accuracy_score(actual_classes, predicted_classes)
# precision = metrics.precision_score(actual_classes, predicted_classes, average='macro')
# recall = metrics.recall_score(actual_classes, predicted_classes, average='macro')
# f1score = metrics.f1_score(actual_classes, predicted_classes, average='macro')

# print("\nConfusion Matrix: \n", conf_matrix)
# print("------------------------------------------------------------")
# print(classification_report(actual_classes, predicted_classes, digits = 4,
#                            target_names = ['Positive', 'Negative', 'Neutral']))
# print("------------------------------------------------------------")
# print("\nAccuracy : %.3f, Precission : %.3f, Recall : %.3f, F1 Score : %.3f" %(svm_accuracy, precision, recall, f1score))

## RNN Clasifier

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding
from keras_preprocessing.sequence import pad_sequences

In [ ]:
# Obtain the embedding matrix for your vocabulary
# embedding_matrix = model.wv.vectors

In [ ]:
# Pad the sequences to a fixed length
# max_length = max(len(sequence) for sequence in X)
# padded_sequences = pad_sequences(X, maxlen=max_length, padding='post')